# H3a: staranność bio odbiorcy a prawy swipe

Parametryzowany notebook (papermill). Logika: `src/rentfriends/h3a/`. Specyfikacja:
`docs/prompts/h3a-notebook-prompt.md`; decyzje: `docs/h3a/decision-log.md`.

**Prywatność:** notebook wyświetla i zapisuje wyłącznie agregaty (komórki < 5 użytkowników
wytłumione). Tekst bio, `user_key` ani wiersze tabel nie są nigdzie pokazywane.

Wnioski dotyczą **związku**, nie przyczynowości; match ≠ najem.

In [1]:
# Parameters (papermill)
RUN_LEVEL = 1  # 3 = poziom 3 (diagnostyka, SE dla GLMM, figury)
SAMPLE_FRAC = 1.0
SAMPLE_BY = "sender"
SEED = 20261102
MARGIN = 0.10
FIT_TIMEOUT_MIN = 90
REPORT_DIR = "reports/h3a"
ONLY = ""  # poziom 3: "" | diagnostics | se | figures | lme4
RUN_BOOTSTRAP = 0  # poziom 3, B3 (bootstrap po nadawcach; noc)
RUN_AUDIT = 1  # poziom 3: sekcja 12 (audyt post hoc Z1-Z8); fity z cache, wynik odświeżany

In [2]:
# Parameters
RUN_LEVEL = 3
SAMPLE_FRAC = 1.0
REPORT_DIR = "reports/h3a"
ONLY = ""
RUN_BOOTSTRAP = 0

In [3]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from rentfriends.h3a import analysis as A
from rentfriends.h3a.summary import write_summary

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
LEGACY = not (RUN_LEVEL >= 3 and ONLY)  # ONLY pomija kroki poziomów 1-2
assert SAMPLE_BY == "sender", "only sender-level sampling is implemented"
ctx = A.setup(
    A.Params(
        run_level=RUN_LEVEL,
        sample_frac=SAMPLE_FRAC,
        sample_by=SAMPLE_BY,
        seed=SEED,
        margin=MARGIN,
        fit_timeout_min=FIT_TIMEOUT_MIN,
        report_dir=REPORT_DIR,
    )
)

## 1. Dane, reguła as-of (D7), przepływ próby (F0) i kontrola zgodności z kartą M1

In [4]:
flow = A.prepare(ctx)
display(flow)
display(pd.DataFrame(ctx.res["card_check"]))
print("Hyderabad:", ctx.res["hyderabad"])
print("Źródło wersji as-of:", ctx.res["asof_source_counts"])
print("G18 (bio_length vs tekst):", ctx.res["g18_bio_length_mismatch"])
print("Braki po filtrze Hyderabad:", ctx.res["missing_by_var_after_hyd"])

<Figure size 1000x560 with 1 Axes>

,krok,swipes,prawe,nadawcy,odbiorcy
0,Swipe'y na profile,95144,19495,1583,1394
1,Po wykluczeniu Hyderabad (nadawca lub odbiorca),94295,19363,1544,1363
2,Po complete-case (zmienne modelu),93717,19325,1538,1359


,miara,karta,asof,odchylenie_pct,bieżący_users,flaga_>1pct
0,profile_swipes,95144.00,95144.000,0.000,NaN,False
1,right_swipes,19495.00,19495.000,0.000,NaN,False
2,right_share_pct,20.49,20.490,-0.000,NaN,False
3,swipers,1583.00,1583.000,0.000,NaN,False
4,users,2065.00,2065.000,0.000,NaN,False
5,swipes_with_bio,52371.00,52297.000,-0.141,52371.000,False
6,right_pct_with_bio,21.50,21.420,-0.372,21.537,False
7,right_pct_without_bio,19.20,19.355,0.807,19.208,False
8,missing_age_band_swipes,578.00,578.000,0.000,NaN,False


Hyderabad: {'accounts': 45, 'accounts_any_version': 45, 'profile_swipes_removed': 849, 'as_sender': 669, 'as_receiver': 627}
Źródło wersji as-of: {'s': {'prior_version': 50932, 'first_version': 44086, 'users_current': 126}, 'r': {'prior_version': 88848, 'first_version': 5625, 'users_current': 671}, 'pair': {'prior_version': 47540, 'first_version': 46846, 'users_current': 758}}
G18 (bio_length vs tekst): {'receiver_snapshots': 2195, 'share_mismatch': 0.020045558086560365, 'share_swipes_mismatch': 0.025718910283359958}
Braki po filtrze Hyderabad: {'age_band_diff': 578, 'same_band': 578}


## 2. Statystyki opisowe (F1, F2, F8)

In [5]:
if LEGACY:
    A.describe(ctx)
    print(ctx.res["desc"])

<Figure size 1000x560 with 1 Axes>

<Figure size 1000x560 with 1 Axes>

<Figure size 1000x560 with 1 Axes>

{'share_empty_swipes': np.float64(0.4498863599987195), 'share_cap_among_nonempty': 0.01737949762389681, 'bio_len_median_nonempty': 95.0, 'bio_len_iqr_nonempty': [50.0, 161.0]}


## 3. Sanity check GLMM na danych symulowanych

Rzeczywisty design (X, nadawcy, odbiorcy), znane parametry: β_bio = 0,15, σ²_nadawca = 1,5,
σ²_odbiorca = 0,5. Estymator powinien je odtworzyć, a 95% CI pokrywać prawdę w ~95% powtórzeń.

In [6]:
if LEGACY:
    A.sanity_check(ctx)

## 4. Modele główne M1 (GLMM), M2 (pooled logit, klastrowanie 2-kier.), M3 (FE nadawcy)
Etykieta według reguły z sekcji 7 promptu (ustalonej przed wynikami).

In [7]:
if LEGACY:
    main_tab = A.main_models(ctx)
    display(main_tab)
    print(ctx.res["verdict"])
    print(ctx.res["m1_diag"])
    print(ctx.res["m3_diag"])

,model,status,beta,se,or,or_lo95,or_hi95,or_lo90,or_hi90,p,mde_or,label
0,M1,not_converged_or_no_se,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania
1,M2,ok,0.122125,0.040187,1.129895,1.044313,1.222491,1.057622,1.207107,0.002374,1.119099,potwierdzony
2,M3,ok,0.172745,0.053581,1.188563,1.070074,1.320173,1.088294,1.298070,0.001264,1.161867,potwierdzony


{'label_M1': 'brak oszacowania', 'label_M2': 'potwierdzony', 'label_M3': 'potwierdzony', 'final': 'potwierdzony (fallback: M2 jako model główny, bez kontroli popularności)', 'lme4': 'nie uruchomiony', 'ame_M2': {'ame_pp': 1.9014251377286537, 'se_pp': 0.6155227818247532, 'lo95_pp': 0.6950226441722834, 'hi95_pp': 3.107827631285024}}
{'status': 'not_converged_or_no_se', 'optimizer': "lbfgs {'delta_rel_conv': 1e-08}", 'convergence_status': 2, 'n_iter': 27, 'time_s': 8.462459087371826, 'var_sender': 7.7698976897200325, 'var_receiver': 0.5592802217717461, 'var_sender_se': 0.4018941360371455, 'var_receiver_se': 0.03314457887706699, 'loglik': -27845.495884444972, 'attempts': [{'optimizer': 'lbfgs', 'convergence_status': 0, 'time_s': 6.481233358383179}, {'optimizer': 'gradient_descent', 'convergence_status': 0, 'time_s': 31.447845220565796}, {'optimizer': "gradient_descent {'delta_rel_conv': 1e-08}", 'convergence_status': 0, 'time_s': 52.702144622802734}, {'optimizer': "lbfgs {'delta_rel_conv':

## 5. M5: długość bio (drugorzędne, bez werdyktu; F5)

In [8]:
if LEGACY:
    display(A.m5_length(ctx))

<Figure size 1000x560 with 1 Axes>

,model,beta_log1p_len,se,or_doubling,or_doubling_lo95,or_doubling_hi95,status,or_cap,or_cap_lo95,or_cap_hi95
0,M1,0.127468,NaN,1.092375,NaN,NaN,not_converged_or_no_se,0.942022,NaN,NaN
1,M2,0.044085,0.019438,1.031029,1.004159,1.058618,ok,0.912061,0.689186,1.207012


## 6. M6: przyrost predykcyjny (opisowy; F7)

In [9]:
if LEGACY:
    m6 = A.m6_predictive(ctx)
    display({k: v for k, v in m6.items() if not k.startswith("_")})

<Figure size 1400x560 with 3 Axes>

{'status': 'ok',
 'n_splits': 5,
 'n_boot': 1000,
 'logloss_base': 0.48846455871879096,
 'logloss_full': 0.48821782992778845,
 'auc_base': 0.6330933958874126,
 'auc_full': 0.6365218721093826,
 'd_logloss': -0.0002467287910025151,
 'd_logloss_lo95': -0.0005069151912649977,
 'd_logloss_hi95': -4.208206212134868e-06,
 'd_auc': 0.003428476221969956,
 'd_auc_lo95': 0.001887189041135301,
 'd_auc_hi95': 0.004987293913791978,
 'aucpr_base': 0.2890285943788405,
 'aucpr_full': 0.2915318420952159,
 'prevalence': 0.2062059178164047,
 'd_aucpr': 0.0025032477163753897,
 'd_aucpr_lo95': 0.0005618555634413436,
 'd_aucpr_hi95': 0.004149106467204539}

## 7. M4: analiza w obrębie odbiorcy z bramką wykonalności (F6)

In [10]:
if LEGACY:
    m4 = A.m4_within(ctx)
    display(m4)

<Figure size 1000x560 with 1 Axes>

{'funnel': {'receivers_in_sample': 1359,
  'with_bio_text_change': 291,
  'with_status_change': 128,
  'swipes_on_status_changers': 11696,
  'swipes_after_exclusions': 10865,
  'swipes_eligible_receivers': 2179,
  'with_min_swipes_each_side': 17,
  'gate_threshold': 30,
  'gate_passed': False,
  'clean_only': False},
 'funnel_clean': {'receivers_in_sample': 1359,
  'with_bio_text_change': 291,
  'with_status_change': 65,
  'swipes_on_status_changers': 5927,
  'swipes_after_exclusions': 5476,
  'swipes_eligible_receivers': 1170,
  'with_min_swipes_each_side': 9,
  'gate_threshold': 30,
  'gate_passed': False,
  'clean_only': True},
 'verdict': 'niemożliwe do zmierzenia tym projektem: 17 odbiorców < 30'}

## 8. Analizy wrażliwości S1-S8 (M2 i M3; na poziomie 2 także GLMM dla S1, S3, S4, S7; F4)

In [11]:
if LEGACY:
    sens = A.sensitivity(ctx, level=1)
    display(sens)
    print(ctx.res["s1_counts"])
    print(ctx.res["s8"]["interaction"])

<Figure size 1000x560 with 1 Axes>

,spec,model,status,n_obs,n_receivers,beta,se,or,or_lo95,or_hi95,or_lo90,or_hi90,p,label
0,S1,M1,not_converged_or_no_se,76187,1273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania
1,S1,M2,ok,76187,1273,0.129659,0.042934,1.138440,1.046560,1.238386,1.060815,1.221744,0.002528,potwierdzony
2,S1,M3,ok,76187,1273,0.173842,0.053818,1.189868,1.070751,1.322236,1.089064,1.300002,0.001237,potwierdzony
3,S2a,M2,ok,47373,1259,0.062101,0.046210,1.064070,0.971932,1.164942,0.986188,1.148102,0.178986,nierozstrzygający
4,S2a,M3,ok,47373,1259,0.160162,0.060746,1.173701,1.041959,1.322101,1.062096,1.297034,0.008374,potwierdzony
5,S2b,M2,ok,93717,1359,0.119265,0.039828,1.126669,1.042065,1.218142,1.055225,1.202950,0.002749,potwierdzony
6,S2b,M3,ok,93717,1359,0.158561,0.055450,1.171824,1.051146,1.306356,1.069674,1.283728,0.004243,potwierdzony
7,S3,M1,not_converged_or_no_se,94566,1390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania
8,S3,M2,ok,94566,1390,0.120306,0.039918,1.127842,1.042965,1.219626,1.056167,1.204381,0.002580,potwierdzony
9,S3,M3,ok,94566,1390,0.171553,0.053335,1.187147,1.069315,1.317963,1.087438,1.295998,0.001298,potwierdzony


{'senders_main': 1538, 'senders_ge10': 1134, 'all_right_ge10': 15, 'all_left_ge10': 328, 'senders_kept': 791}
{'terms': ['has_bio:sender_sex=M', 'has_bio:sender_sex=O'], 'coef': [-0.18418900355795503, -0.10854758179858442], 'se': [0.06637274287019969, 0.18988734325491502], 'wald_chi2': 7.821730401806833, 'df': 2, 'p': 0.02002316950078277}


## 9. Poziom 2: refity GLMM dla S1, S3, S4, S7 i cross-check lme4

In [12]:
if LEGACY:
    if RUN_LEVEL >= 2:
        sens = A.sensitivity(ctx, level=2)
        display(sens[sens["model"] == "M1"])
    print(A.lme4_crosscheck(ctx))

<Figure size 1000x560 with 1 Axes>

,spec,model,status,n_obs,n_receivers,beta,se,or,or_lo95,or_hi95,or_lo90,or_hi90,p,label
0,S1,M1,not_converged_or_no_se,76187,1273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania
7,S3,M1,not_converged_or_no_se,94566,1390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania
10,S4,M1,not_converged_or_no_se,93717,1359,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania
17,S7,M1,not_converged_or_no_se,93717,1359,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania


{'status': 'ok', 'names': ['const', 'has_bio', 'age_band_diff', 'same_band', 'profile_completeness', 'tag_count_receiver', 'tag_zero', 'tag_cap', 'sex_pair=F→M', 'sex_pair=F→O', 'sex_pair=M→F', 'sex_pair=M→M', 'sex_pair=M→O', 'sex_pair=O→F', 'sex_pair=O→M', 'sex_pair=O→O', 'same_city=different', 'same_city=unknown'], 'coef': [-2.73753406364227, 0.124010901334206, -0.403397553914904, 0.551375708817852, -0.0962153132539259, 0.138231283188529, 0.413905284423284, -0.0693629583319441, -1.47550800024057, -1.22810361950449, 0.229447149433901, -0.6010611227054, -0.758289780253925, -0.783693202698573, -0.633725470749262, 0.343241110218211, -0.128446197719659, 0.128252223825735], 'se': [0.12055108838144, 0.0492821672891129, 0.0353508327997915, 0.0309148699441081, 0.0233222565705557, 0.0395287235728356, 0.269914297808174, 0.074910527355796, 0.0658734962243268, 0.192176310567416, 0.151925888593989, 0.159641277186812, 0.241553046295642, 0.316273867714575, 0.343688972862147, 0.782151732813204, 0.097

## 10. Wykres główny (F3) i podsumowanie

In [13]:
if LEGACY:
    forest_df = A.forest(ctx)
    path = write_summary(ctx.res, REPORT_DIR, RUN_LEVEL)
    print(path.read_text())

<Figure size 1000x560 with 1 Axes>

# H3a: wyniki (generowane automatycznie)

Wygenerowano: 2026-11-10T03:28; poziom przebiegu: 3; SAMPLE_FRAC = 1.0; SEED = 20261102; margines praktycznego zera |log OR| ≤ 0,10.

## Werdykt

**Etykieta końcowa: potwierdzony (fallback: M2 jako model główny, bez kontroli popularności)**

- M1 (GLMM, skrzyżowane efekty losowe): brak oszacowania
- M2 (pooled logit, klastrowanie dwukierunkowe): potwierdzony
- M3 (FE nadawcy, klastrowanie po odbiorcy; poza regułą): potwierdzony
- lme4: ok; etykieta potwierdzony (zgodna z M2: tak)

## Wynik główny (has_bio: bio odbiorcy niepuste vs puste)

| Model | OR (95% CI; 90% CI) | β | SE | p | MDE OR | Etykieta |
|---|---|---|---|---|---|---|
| M1 | – (95% CI –––; 90% CI –––) | – | – | – | – | brak oszacowania |
| M2 | 1,130 (95% CI 1,044–1,222; 90% CI 1,058–1,207) | 0,1221 | 0,0402 | 0,0024 | 1,119 | potwierdzony |
| M3 | 1,189 (95% CI 1,070–1,320; 90% CI 1,088–1,298) | 0,1727 | 0,0536 | 0,0013 | 1,162 | potwierdzony |

**M1 raportowany = lme4** (Wald): 

## 11. Poziom 3: diagnostyka post hoc (A1-A5), SE dla GLMM (B), figury i eksport (C)

Parametry `ONLY` i `RUN_BOOTSTRAP`; kolejność: A5, figury, A1-A4, B1, B2. Wynik: `reports/h3a/summary_level3.md`, `reports/h3a/export/`.

In [14]:
if RUN_LEVEL >= 3:
    from rentfriends.h3a import level3 as L3

    files = L3.run_level3(ctx, only=ONLY, bootstrap=bool(RUN_BOOTSTRAP))
    print(files)

['h3a_captions.md', 'h3a_fig_A.png', 'h3a_fig_A.svg', 'h3a_fig_A_preview.png', 'h3a_fig_B.png', 'h3a_fig_B.svg', 'h3a_fig_B_preview.png', 'h3a_fig_C.png', 'h3a_fig_C.svg', 'h3a_fig_C_preview.png', 'h3a_numbers.json', 'theme.json']


## 12. Audyt po wynikach (post hoc, `post hoc (audyt)`): Z1-Z8

Reguła etykiet z sekcji 7 pozostaje bez zmian. Poniżej: ślad reguły (Z1), M1 z lme4 (Z2),
płeć nadawcy (Z3), stabilność bio i as-of (Z4), kontrole i słownik tagów v1 (Z5), AUC-PR (Z6),
zmienne jakości bio bez werdyktu (Z8). Fity są w cache (`data/interim/h3a/`, poza gitem);
wynik zbiorczy: `reports/h3a/audit_results.json`. Wykresy czytają tylko ten plik
(`rentfriends.h3a.audit_plots`).

In [15]:
AUDIT_ON = bool(RUN_AUDIT) and RUN_LEVEL >= 3 and not ONLY and SAMPLE_FRAC >= 1.0
if AUDIT_ON:
    from rentfriends.h3a import audit as AU
    from rentfriends.h3a import audit_plots as AP

    audit = AU.run_audit(refresh=True)
    print("N (próba główna):", audit["n_main"])
else:
    print("Sekcja audytu pominięta (RUN_AUDIT, RUN_LEVEL < 3, ONLY lub SAMPLE_FRAC < 1).")

N (próba główna): 93717


### Z1. Ślad reguły etykiety (M2, M3, M1 = lme4)

In [16]:
if AUDIT_ON:
    keep = [
        "beta",
        "se",
        "or",
        "or_lo95",
        "or_hi95",
        "or_lo90",
        "or_hi90",
        "equiv",
        "sig_pos_lo95_gt_0",
        "lo95_above_margin",
        "label",
    ]
    display(pd.DataFrame(audit["z1_trace"]).T[keep])

,beta,se,or,or_lo95,or_hi95,or_lo90,or_hi90,equiv,sig_pos_lo95_gt_0,lo95_above_margin,label
M2,0.122125,0.040187,1.129895,1.044313,1.222491,1.057622,1.207107,False,True,False,potwierdzony
M3,0.172745,0.053581,1.188563,1.070074,1.320173,1.088294,1.29807,False,True,False,potwierdzony
M1_lme4,0.124011,0.049282,1.132028,1.027799,1.246827,1.043885,1.227614,False,True,False,potwierdzony


### Z2. M1: lme4 jako raportowany estymator (GPBoost tylko punkt)

In [17]:
if AUDIT_ON:
    display(pd.DataFrame(audit["z2_table"]))
    print(audit["z2_lme4_diag"])

,model,beta,se,or,p,or_lo95,or_hi95,or_lo90,or_hi90,lrt_p,label,status,mde_or
0,"M1 (lme4, Wald)",0.124011,0.049282,1.132028,0.011858,1.027799,1.246827,1.043885,1.227614,0.008963,potwierdzony,NaN,NaN
1,"M1 (GPBoost, tylko punkt)",0.130423,NaN,1.139310,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania (brak SE),NaN,NaN
2,M2,0.122125,0.040187,1.129895,0.002374,1.044313,1.222491,1.057622,1.207107,NaN,potwierdzony,ok,1.119099
3,M3,0.172745,0.053581,1.188563,0.001264,1.070074,1.320173,1.088294,1.298070,NaN,potwierdzony,ok,1.161867


{'var_sender': 7.43259783422814, 'var_receiver': 0.557219409302682, 'singular': False, 'conv_code': 0, 'r_warnings': [], 'loglik': -27864.7766589834}


### Z3-Z5. Płeć nadawcy (eksploracyjnie), stabilność bio / as-of, kontrole

In [18]:
if AUDIT_ON:
    print(audit["z4_counts"])
    print(audit["z5_tags_v1_check"])
    for sec in ("z3", "z4", "z5"):
        display(AP.scenario_table(audit, sec))
        AP.plot_scenarios(sec, kind="forest", res=audit, show=True)

{'receivers': 1359, 'with_text_change': 291, 'stable_text': 1068, 'with_status_change': 128}
{'same_sample_size': True, 'n_v0': 93717, 'n_v1': 93717, 'share_swipes_tag_count_changed': 0.10285220397579949, 'mean_tag_count_v0': 7.518849301620837, 'mean_tag_count_v1': 7.4153141905950894, 'tag_zero_changed': 0, 'tag_cap_changed': 6307}


,spec,model,estimator,n_obs,n_receivers,or,or_lo95,or_hi95,or_lo90,or_hi90,p,label
0,Z3_F,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",50241,1286,1.258832,1.132552,1.399192,1.151965,1.375613,0.000020,potwierdzony
1,Z3_F,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",50241,1286,1.307424,1.144638,1.493362,1.169372,1.461775,0.000078,potwierdzony
2,Z3_F,M1_lme4,"M1 (lme4, Wald)",50241,1286,1.253211,1.107054,1.418665,1.129346,1.390661,0.000361,potwierdzony
3,Z3_M,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",42621,1331,1.030896,0.936782,1.134464,0.951312,1.117137,0.533311,nierozstrzygający
4,Z3_M,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",42621,1331,1.088927,0.973340,1.218241,0.991060,1.196459,0.136750,nierozstrzygający
5,Z3_M,M1_lme4,"M1 (lme4, Wald)",42621,1331,1.068877,0.956397,1.194585,0.973648,1.173420,0.240352,nierozstrzygający
6,Z3_O,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",855,604,1.267974,0.905047,1.776435,0.955465,1.682696,0.167573,nierozstrzygający
7,Z3_O,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",855,604,NaN,NaN,NaN,NaN,NaN,NaN,brak oszacowania


<Figure size 700x395 with 1 Axes>

,spec,model,estimator,n_obs,n_receivers,or,or_lo95,or_hi95,or_lo90,or_hi90,p,label
0,Z4_stable_text,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",68906,1068,1.131715,1.037749,1.234191,1.052312,1.217110,0.005145,potwierdzony
1,Z4_stable_text,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",68906,1068,1.176105,1.038248,1.332266,1.059269,1.305828,0.010771,potwierdzony
2,Z4_stable_text,M1_lme4,"M1 (lme4, Wald)",68906,1068,1.192884,1.057766,1.345262,1.078409,1.319511,0.004033,potwierdzony
3,Z4_no_status_change,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",82021,1231,1.142793,1.053120,1.240102,1.067047,1.223916,0.001368,potwierdzony
4,Z4_no_status_change,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",82021,1231,1.202164,1.072294,1.347763,1.092185,1.323217,0.001596,potwierdzony
5,Z4_no_status_change,M1_lme4,"M1 (lme4, Wald)",82021,1231,1.233478,1.103859,1.378316,1.123740,1.353932,0.000212,potwierdzony
6,Z4_S2a_prior_version,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",47373,1259,1.064070,0.971932,1.164942,0.986188,1.148102,0.178986,nierozstrzygający
7,Z4_S2a_prior_version,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",47373,1259,1.173701,1.041959,1.322101,1.062096,1.297034,0.008374,potwierdzony
8,Z4_S2a_prior_version,M1_lme4,"M1 (lme4, Wald)",47373,1259,1.127778,1.001916,1.269451,1.021160,1.245528,0.046408,potwierdzony


<Figure size 700x465 with 1 Axes>

,spec,model,estimator,n_obs,n_receivers,or,or_lo95,or_hi95,or_lo90,or_hi90,p,label
0,Z5_age_h1,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",93717,1359,1.137324,1.050929,1.230821,1.064362,1.215286,0.001411,potwierdzony
1,Z5_age_h1,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",93717,1359,1.197619,1.078513,1.329879,1.096831,1.307670,0.000740,potwierdzony
2,Z5_age_h1,M1_lme4,"M1 (lme4, Wald)",93717,1359,1.143087,1.037782,1.259076,1.054034,1.239664,0.006687,potwierdzony
3,Z5_activity,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",93717,1359,1.096703,1.015710,1.184156,1.028316,1.169639,0.018365,potwierdzony
4,Z5_activity,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",93717,1359,1.120909,1.012419,1.241025,1.029125,1.220880,0.027978,potwierdzony
5,Z5_activity,M1_lme4,"M1 (lme4, Wald)",93717,1359,1.093496,0.992060,1.205303,1.007710,1.186585,0.071944,nierozstrzygający
6,Z5_age_h1_activity,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",93717,1359,1.105565,1.023757,1.193911,1.036489,1.179245,0.010511,potwierdzony
7,Z5_age_h1_activity,M3,"M3 (FE nadawcy, klastrowanie po odbiorcy)",93717,1359,1.130373,1.021269,1.251134,1.038071,1.230883,0.017965,potwierdzony
8,Z5_age_h1_activity,M1_lme4,"M1 (lme4, Wald)",93717,1359,1.105083,1.002482,1.218184,1.018311,1.199248,0.044449,potwierdzony
9,Z5_tags_v1,M2,"M2 (pooled logit, klastrowanie dwukierunkowe)",93717,1359,1.129943,1.044228,1.222694,1.057557,1.207284,0.002404,potwierdzony


<Figure size 700x570 with 1 Axes>

### Z6. M6: AUC-PR z bootstrapem po nadawcach (opisowo)

In [19]:
if AUDIT_ON:
    m6 = ctx.res["m6"]
    print({k: m6[k] for k in m6 if "aucpr" in k or k == "prevalence"})

{'aucpr_base': 0.2890285943788405, 'aucpr_full': 0.2915318420952159, 'prevalence': 0.2062059178164047, 'd_aucpr': 0.0025032477163753897, 'd_aucpr_lo95': 0.0005618555634413436, 'd_aucpr_hi95': 0.004149106467204539}


### Z8. Zmienne jakości bio (emoji, `?`, język): opisowo, bez werdyktu

In [20]:
if AUDIT_ON:
    z8 = audit["z8"]
    print(z8["note"], "| N swipe'ów z bio:", z8["n_obs"])
    display(pd.Series(z8["desc"]).to_frame("wartość"))
    display(pd.DataFrame(z8["terms"])[["term", "model", "or", "or_lo95", "or_hi95", "p"]])

opisowo, bez werdyktu; język: nieprzetestowana heurystyka stop-słów | N swipe'ów z bio: 51555


,wartość
swipes_with_bio,51555.000000
receivers_with_bio,768.000000
share_swipes_any_emoji,0.183202
share_swipes_any_question,0.001629
share_swipes_lang_pl,0.844419
share_swipes_lang_en,0.054718
share_swipes_lang_other,0.100863
right_rate_any_emoji,0.242562
right_rate_no_emoji,0.209784
right_rate_any_question,0.214286


,term,model,or,or_lo95,or_hi95,p
0,log1p_bio_len,bez zmiennych jakości,1.045071,1.006005,1.085654,0.023330
1,log1p_bio_len,ze zmiennymi jakości,1.032549,0.975403,1.093044,0.270187
2,bio_len_cap,bez zmiennych jakości,0.912061,0.689186,1.207012,0.519657
3,bio_len_cap,ze zmiennymi jakości,0.919224,0.687727,1.228645,0.569376
4,log1p_emoji,ze zmiennymi jakości,1.057650,0.933497,1.198314,0.378980
5,log1p_question,ze zmiennymi jakości,1.306587,0.452409,3.773512,0.621173
6,lang_pl,ze zmiennymi jakości,1.051010,0.875624,1.261526,0.593272
7,lang_en,ze zmiennymi jakości,0.899611,0.693585,1.166838,0.425322


## 13. Summary po audycie

`summary.md` jest generowany ponownie po odświeżeniu `audit_results.json` (sekcja 12).

In [21]:
if AUDIT_ON:
    path = write_summary(ctx.res, REPORT_DIR, RUN_LEVEL)
    print(path.read_text())

# H3a: wyniki (generowane automatycznie)

Wygenerowano: 2026-11-10T03:29; poziom przebiegu: 3; SAMPLE_FRAC = 1.0; SEED = 20261102; margines praktycznego zera |log OR| ≤ 0,10.

## Werdykt

**Etykieta końcowa: potwierdzony (fallback: M2 jako model główny, bez kontroli popularności)**

- M1 (GLMM, skrzyżowane efekty losowe): brak oszacowania
- M2 (pooled logit, klastrowanie dwukierunkowe): potwierdzony
- M3 (FE nadawcy, klastrowanie po odbiorcy; poza regułą): potwierdzony
- lme4: ok; etykieta potwierdzony (zgodna z M2: tak)

## Wynik główny (has_bio: bio odbiorcy niepuste vs puste)

| Model | OR (95% CI; 90% CI) | β | SE | p | MDE OR | Etykieta |
|---|---|---|---|---|---|---|
| M1 | – (95% CI –––; 90% CI –––) | – | – | – | – | brak oszacowania |
| M2 | 1,130 (95% CI 1,044–1,222; 90% CI 1,058–1,207) | 0,1221 | 0,0402 | 0,0024 | 1,119 | potwierdzony |
| M3 | 1,189 (95% CI 1,070–1,320; 90% CI 1,088–1,298) | 0,1727 | 0,0536 | 0,0013 | 1,162 | potwierdzony |

**M1 raportowany = lme4** (Wald): 